In [3]:
from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import Chroma
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
import os


# Clone the Github Repo

In [4]:
!mkdir -p test_repo

In [7]:
repo_path = 'test_repo'
repo = Repo.clone_from("https://github.com/miteshupadhyay/Demo_YouTube_Content_Creation_Agent", to_path = repo_path)

# Load the data from project working directory

In [8]:
%pwd

'd:\\Personal\\Final_GenAI\\Demo_Source_Code_Analyzer\\Demo_Source_Code_Analyzer\\research'

In [11]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob="**/*",
                                       suffixes=[".py"],
                                       parser=LanguageParser(language=Language.PYTHON, parser_threshold=500))

In [12]:
document = loader.load()

In [13]:
document

[Document(page_content='import streamlit as st\nimport os\nfrom dotenv import load_dotenv\nfrom google import genai\nfrom tavily import TavilyClient\nimport time\nfrom google.genai import types\nfrom moviepy import VideoFileClip\n\nload_dotenv()\n\n# Configure API keys\nGEMINI_API_KEY = os.getenv("GEMINI_API_KEY")\nTAVILY_API_KEY = os.getenv("TAVILY_API_KEY")\n\nif not GEMINI_API_KEY:\n    st.error("GEMINI_API_KEY is missing from the .env file.")\n    st.stop()\n\nif not TAVILY_API_KEY:\n    st.error("TAVILY_API_KEY is missing from the .env file.")\n    st.stop()\n\n# Initialize Gemini and Tavily clients\ngeni_client = genai.Client(api_key=GEMINI_API_KEY)\ntavily_client = TavilyClient(api_key=TAVILY_API_KEY)\n\n# Select models\nMODEL_INFO = "gemini-3.8-flash"\nMODEL_SCRIPT = "gemini-3.8-flash"\n\n\n# Streamlit configuration\nst.set_page_config(\n    page_title="StoryForge Agent",\n    page_icon="🌐",\n    layout="centered",\n    initial_sidebar_state="collapsed"\n)\n\n\n# Custom modern 

In [14]:
len(document)

3

In [15]:
document[0]

Document(page_content='import streamlit as st\nimport os\nfrom dotenv import load_dotenv\nfrom google import genai\nfrom tavily import TavilyClient\nimport time\nfrom google.genai import types\nfrom moviepy import VideoFileClip\n\nload_dotenv()\n\n# Configure API keys\nGEMINI_API_KEY = os.getenv("GEMINI_API_KEY")\nTAVILY_API_KEY = os.getenv("TAVILY_API_KEY")\n\nif not GEMINI_API_KEY:\n    st.error("GEMINI_API_KEY is missing from the .env file.")\n    st.stop()\n\nif not TAVILY_API_KEY:\n    st.error("TAVILY_API_KEY is missing from the .env file.")\n    st.stop()\n\n# Initialize Gemini and Tavily clients\ngeni_client = genai.Client(api_key=GEMINI_API_KEY)\ntavily_client = TavilyClient(api_key=TAVILY_API_KEY)\n\n# Select models\nMODEL_INFO = "gemini-3.8-flash"\nMODEL_SCRIPT = "gemini-3.8-flash"\n\n\n# Streamlit configuration\nst.set_page_config(\n    page_title="StoryForge Agent",\n    page_icon="🌐",\n    layout="centered",\n    initial_sidebar_state="collapsed"\n)\n\n\n# Custom modern C

# Perform Context Aware Chunking

In [16]:
document_spllitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
                                                                  chunk_size = 500,
                                                                  chunk_overlap = 20)

In [17]:
texts = document_spllitter.split_documents(document)

In [18]:
len(texts)

39

In [19]:
texts

[Document(page_content='import streamlit as st\nimport os\nfrom dotenv import load_dotenv\nfrom google import genai\nfrom tavily import TavilyClient\nimport time\nfrom google.genai import types\nfrom moviepy import VideoFileClip\n\nload_dotenv()\n\n# Configure API keys\nGEMINI_API_KEY = os.getenv("GEMINI_API_KEY")\nTAVILY_API_KEY = os.getenv("TAVILY_API_KEY")\n\nif not GEMINI_API_KEY:\n    st.error("GEMINI_API_KEY is missing from the .env file.")\n    st.stop()', metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}),
 Document(page_content='if not TAVILY_API_KEY:\n    st.error("TAVILY_API_KEY is missing from the .env file.")\n    st.stop()\n\n# Initialize Gemini and Tavily clients\ngeni_client = genai.Client(api_key=GEMINI_API_KEY)\ntavily_client = TavilyClient(api_key=TAVILY_API_KEY)\n\n# Select models\nMODEL_INFO = "gemini-3.8-flash"\nMODEL_SCRIPT = "gemini-3.8-flash"\n\n\n# Streamlit configuration\nst.set_page_config(\n    page_title="StoryForge Agent",\

# Download the OpenAI Embedding

In [21]:
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
OPENAI_API_KEY= os.environ.get('OPENAI_API_KEY')

In [23]:
os.environ["OPENAI_API_KEY"]= OPENAI_API_KEY

In [24]:
embeddings = OpenAIEmbeddings(disallowed_special=())

# Setting up the vector Database

In [25]:
vectordb = Chroma.from_documents(texts, embedding= embeddings, persist_directory='./data')

ValueError: Could not import chromadb python package. Please install it with `pip install chromadb`.

In [ ]:
vectordb.persist()

# Creating an OpenAI Model Wrapper

In [26]:
llm = ChatOpenAI()

In [27]:
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

In [28]:
qa= ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type ="mmr", search_kwargs={"k":8}), memory=memory)

NameError: name 'vectordb' is not defined

# Question and Answer

In [ ]:
question = "What is get_realtime_info function"

In [ ]:
result = qa(question)

In [ ]:
print(result)